# YOLOv8 Face Recognition Training Notebook
## Auto-apprentissage pour la reconnaissance faciale en entreprise

Ce notebook permet d'entraîner un modèle YOLOv8 personnalisé pour la reconnaissance faciale.

**Compatible avec:**
- Google Colab
- Jupyter Notebook local
- Azure ML Studio

**Fonctionnalités:**
- Détection de visages
- Détection d'armes
- Détection d'animaux
- Auto-apprentissage sur nouveaux datasets

## 1. Installation des dépendances

In [ ]:
# Installer les packages nécessaires
!pip install ultralytics opencv-python numpy matplotlib albumentations roboflow
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Pour Google Colab, monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports et configuration

In [ ]:
import os
import yaml
import shutil
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from datetime import datetime
import json

# Configuration
ROOT_DIR = Path('/content/drive/MyDrive/YOLO_FaceRecognition')  # Pour Colab
# ROOT_DIR = Path('../')  # Pour local

DATASET_DIR = ROOT_DIR / 'datasets'
MODELS_DIR = ROOT_DIR / 'trained_models'
RESULTS_DIR = ROOT_DIR / 'results'

# Créer les dossiers
for dir_path in [DATASET_DIR, MODELS_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print("Configuration terminée!")
print(f"Dossier racine: {ROOT_DIR}")

## 3. Préparation du dataset

### Structure du dataset attendue:
```
datasets/
├── train/
│   ├── images/
│   └── labels/
├── val/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

In [ ]:
# Créer la structure du dataset
for split in ['train', 'val', 'test']:
    (DATASET_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

print("Structure du dataset créée!")

## 4. Téléchargement de datasets publics (optionnel)

Utiliser Roboflow pour obtenir des datasets pré-annotés

In [ ]:
# Exemple: Télécharger un dataset de détection de visages depuis Roboflow
# Remplacer avec votre propre clé API Roboflow

from roboflow import Roboflow

# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("workspace-name").project("project-name")
# dataset = project.version(1).download("yolov8", location=str(DATASET_DIR))

print("Dataset téléchargé (si configuré)")

## 5. Augmentation de données

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Pipeline d'augmentation
augmentation_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.RandomShadow(p=0.3),
    A.ColorJitter(p=0.3),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

def augment_image(image_path, label_path, output_dir, num_augmentations=5):
    """Appliquer l'augmentation à une image."""
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Lire les annotations YOLO
    with open(label_path, 'r') as f:
        annotations = [line.strip().split() for line in f.readlines()]
    
    if not annotations:
        return
    
    bboxes = [[float(x) for x in ann[1:]] for ann in annotations]
    class_labels = [int(ann[0]) for ann in annotations]
    
    # Appliquer l'augmentation
    for i in range(num_augmentations):
        augmented = augmentation_pipeline(image=image, bboxes=bboxes, class_labels=class_labels)
        
        aug_image = augmented['image']
        aug_bboxes = augmented['bboxes']
        aug_labels = augmented['class_labels']
        
        # Sauvegarder l'image augmentée
        output_image_path = output_dir / 'images' / f"{image_path.stem}_aug{i}.jpg"
        output_label_path = output_dir / 'labels' / f"{image_path.stem}_aug{i}.txt"
        
        cv2.imwrite(str(output_image_path), cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR))
        
        # Sauvegarder les annotations
        with open(output_label_path, 'w') as f:
            for label, bbox in zip(aug_labels, aug_bboxes):
                f.write(f"{label} {' '.join(map(str, bbox))}\n")

print("Pipeline d'augmentation configuré!")

## 6. Créer le fichier de configuration du dataset

In [ ]:
# Configuration pour la détection de visages
dataset_config = {
    'path': str(DATASET_DIR),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 1,  # Nombre de classes (1 pour face seulement)
    'names': ['face']
}

# Pour la détection d'armes
weapons_config = {
    'path': str(DATASET_DIR / 'weapons'),
    'train': 'train/images',
    'val': 'val/images',
    'nc': 4,
    'names': ['gun', 'knife', 'rifle', 'scissors']
}

# Sauvegarder les configs
with open(DATASET_DIR / 'face_dataset.yaml', 'w') as f:
    yaml.dump(dataset_config, f)

with open(DATASET_DIR / 'weapons_dataset.yaml', 'w') as f:
    yaml.dump(weapons_config, f)

print("Fichiers de configuration créés!")

## 7. Entraînement du modèle de détection de visages

In [ ]:
# Charger le modèle pré-entraîné YOLOv8
model = YOLO('yolov8n.pt')  # nano pour plus de vitesse, ou yolov8s.pt, yolov8m.pt pour plus de précision

# Paramètres d'entraînement
training_params = {
    'data': str(DATASET_DIR / 'face_dataset.yaml'),
    'epochs': 100,
    'imgsz': 640,
    'batch': 16,
    'name': f'face_detection_{datetime.now().strftime("%Y%m%d_%H%M%S")}',
    'patience': 50,
    'save': True,
    'device': 0,  # GPU 0, utiliser 'cpu' si pas de GPU
    'workers': 8,
    'optimizer': 'Adam',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'degrees': 0.0,
    'translate': 0.1,
    'scale': 0.5,
    'shear': 0.0,
    'perspective': 0.0,
    'flipud': 0.0,
    'fliplr': 0.5,
    'mosaic': 1.0,
    'mixup': 0.0,
    'copy_paste': 0.0
}

print("Démarrage de l'entraînement...")
print(f"Paramètres: {training_params}")

In [ ]:
# Lancer l'entraînement
results = model.train(**training_params)

print("\n" + "="*50)
print("Entraînement terminé!")
print("="*50)

## 8. Évaluation du modèle

In [ ]:
# Évaluer sur le set de validation
metrics = model.val()

print("\nMétriques de validation:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
# Visualiser les courbes d'entraînement
from IPython.display import Image, display

results_dir = Path('runs/detect') / training_params['name']

# Afficher les résultats
for img_name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    img_path = results_dir / img_name
    if img_path.exists():
        print(f"\n{img_name}:")
        display(Image(filename=str(img_path)))

## 9. Test sur de nouvelles images

In [ ]:
# Charger le meilleur modèle
best_model = YOLO(results_dir / 'weights' / 'best.pt')

# Tester sur une image
test_image_path = DATASET_DIR / 'test' / 'images' / 'test_image.jpg'  # Remplacer avec votre image

if test_image_path.exists():
    results = best_model.predict(test_image_path, conf=0.5)
    
    # Afficher les résultats
    for r in results:
        im_array = r.plot()
        im_rgb = cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(12, 8))
        plt.imshow(im_rgb)
        plt.axis('off')
        plt.title('Détection de visages')
        plt.show()
        
        print(f"\nNombre de visages détectés: {len(r.boxes)}")
        for i, box in enumerate(r.boxes):
            print(f"Visage {i+1}: Confiance = {box.conf[0]:.2f}")
else:
    print("Image de test non trouvée")

## 10. Export du modèle

In [ ]:
# Exporter en différents formats
model_name = f"yolov8n-face-{datetime.now().strftime('%Y%m%d')}"

# Format PyTorch
shutil.copy(results_dir / 'weights' / 'best.pt', MODELS_DIR / f'{model_name}.pt')

# Format ONNX (pour déploiement)
best_model.export(format='onnx', dynamic=True)

# Format TensorRT (pour NVIDIA GPUs)
# best_model.export(format='engine', device=0)

print(f"\nModèle exporté: {model_name}.pt")
print(f"Localisation: {MODELS_DIR}")

## 11. Entraînement du modèle de détection d'armes

In [ ]:
# Charger un nouveau modèle pour les armes
weapon_model = YOLO('yolov8n.pt')

# Paramètres d'entraînement pour les armes
weapon_training_params = training_params.copy()
weapon_training_params.update({
    'data': str(DATASET_DIR / 'weapons_dataset.yaml'),
    'name': f'weapon_detection_{datetime.now().strftime("%Y%m%d_%H%M%S")}',
    'epochs': 150  # Plus d'epochs pour la détection d'armes
})

print("Entraînement du modèle de détection d'armes...")
# weapon_results = weapon_model.train(**weapon_training_params)
print("(Décommentez la ligne ci-dessus pour lancer l'entraînement)")

## 12. Auto-apprentissage (Incremental Learning)

In [ ]:
def incremental_training(base_model_path, new_data_path, epochs=50):
    """
    Entraîner le modèle sur de nouvelles données.
    Utile pour l'auto-apprentissage continu.
    """
    # Charger le modèle existant
    model = YOLO(base_model_path)
    
    # Continuer l'entraînement avec les nouvelles données
    results = model.train(
        data=new_data_path,
        epochs=epochs,
        resume=True,  # Continuer depuis le modèle existant
        patience=20,
        name=f'incremental_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    )
    
    return results

print("Fonction d'auto-apprentissage définie!")
print("Utilisez incremental_training() pour entraîner sur de nouvelles données")

## 13. Synchronisation avec Azure

In [ ]:
# Installer Azure SDK
!pip install azure-storage-blob azure-identity

from azure.storage.blob import BlobServiceClient

def upload_model_to_azure(model_path, connection_string, container_name):
    """
    Upload le modèle entraîné vers Azure Blob Storage.
    """
    try:
        blob_service_client = BlobServiceClient.from_connection_string(connection_string)
        
        blob_name = f"models/{Path(model_path).name}"
        blob_client = blob_service_client.get_blob_client(
            container=container_name,
            blob=blob_name
        )
        
        with open(model_path, "rb") as data:
            blob_client.upload_blob(data, overwrite=True)
        
        print(f"Modèle uploadé avec succès: {blob_name}")
        return True
    except Exception as e:
        print(f"Erreur lors de l'upload: {e}")
        return False

# Exemple d'utilisation (à configurer avec vos credentials Azure)
# AZURE_CONNECTION_STRING = "your-connection-string"
# upload_model_to_azure(MODELS_DIR / f'{model_name}.pt', AZURE_CONNECTION_STRING, 'models')

print("Fonction de synchronisation Azure définie!")

## 14. Résumé et sauvegarde des métriques

In [ ]:
# Créer un rapport d'entraînement
training_report = {
    'timestamp': datetime.now().isoformat(),
    'model_name': model_name,
    'model_type': 'face_detection',
    'base_model': 'yolov8n',
    'epochs': training_params['epochs'],
    'batch_size': training_params['batch'],
    'image_size': training_params['imgsz'],
    'metrics': {
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr)
    },
    'model_path': str(MODELS_DIR / f'{model_name}.pt')
}

# Sauvegarder le rapport
report_path = RESULTS_DIR / f'training_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
with open(report_path, 'w') as f:
    json.dump(training_report, f, indent=2)

print("\n" + "="*50)
print("RAPPORT D'ENTRAÎNEMENT")
print("="*50)
print(json.dumps(training_report, indent=2))
print(f"\nRapport sauvegardé: {report_path}")

## 15. Instructions pour le déploiement

### Étapes suivantes:

1. **Copier le modèle vers le serveur:**
   ```bash
   cp /path/to/model.pt /path/to/backend/trained_models/
   ```

2. **Mettre à jour la configuration:**
   - Modifier `backend/.env` avec le nom du nouveau modèle
   - `YOLO_FACE_MODEL=yolov8n-face-20240101.pt`

3. **Redémarrer le serveur:**
   ```bash
   cd backend
   uvicorn app.main:app --reload
   ```

4. **Tester l'API:**
   ```bash
   curl -X POST "http://localhost:8000/api/recognize" \
     -F "file=@test_image.jpg"
   ```

### Pour l'auto-apprentissage:

1. Collecter de nouvelles images
2. Les annoter avec des outils comme LabelImg ou Roboflow
3. Lancer un entraînement incrémental avec `incremental_training()`
4. Évaluer et déployer le nouveau modèle

### Pour Azure ML:

1. Créer un workspace Azure ML
2. Upload ce notebook vers Azure ML Studio
3. Configurer un compute instance (GPU recommandé)
4. Programmer des entraînements automatiques avec Azure ML Pipelines